# Métricas de calidad, costo y rendimiento

Este notebook enseña que la observabilidad no es solo ver logs: también es medir la calidad, la eficiencia y el costo de un agente LLM.

## Objetivos
- Medir latencia, tokens, costo y errores.
- Evaluar calidad mediante una rúbrica simple.
- Comparar dos versiones de prompt.
- Detectar regresiones entre versiones.

In [1]:
!pip install pandas langchain langchain-openai wikipedia python-dotenv

  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\ingri\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


## El caso

Un equipo quiere cambiar su prompt de v1 a v2. Para decidirlo, prueba ambas versiones con las mismas preguntas y compara **calidad**, **rendimiento** y **costo**.

## Dataset de prueba

Usamos 10 preguntas de distintos tipos: conocimiento general, cálculos y datos en tiempo real. Las mismas preguntas se usan para probar ambas versiones del prompt.

In [4]:
import random
import time
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    raise ImportError('Instala pandas con `pip install pandas` antes de ejecutar este notebook.')

# Dataset de 10 preguntas de ejemplo de conocimiento general, calculos, datos en tiempo real 
questions = [
    '¿Cuál es la capital de Argentina?',
    'Explícame qué es la observabilidad en LLMs.',
    'Calcula 12 * 7.',
    'Busca la mejor receta de guacamole.',
    '¿Qué hizo Nelson Mandela?',
    'Resume las ventajas de usar memoria en agentes.',
    'Define qué es un span en trazabilidad.',
    '¿Cómo funciona una batería de auto eléctrico?',
    'Dime un dato sobre el clima actual.',
    '¿Qué es una regresión de prompt?'
]

## Cómo funciona la simulación

1. **Latencia:** v1 ≈ 0.8 s, v2 ≈ 1.1 s, con variación aleatoria.
2. **Tokens:** se cuentan palabras de la pregunta (aproximación, no tokens reales).
3. **Costo:** más tokens = más costo.
4. **Precisión y fundamentación:** valores elegidos al azar de una lista.
5. **Rúbrica:** una respuesta es útil si precisión y fundamentación son ≥ 0.7.

## Lo que esto nos enseña

1. Una métrica solo sirve si se mide bien.
2. Aquí los números salen de reglas escritas por el programador, no de respuestas reales.
3. Si cambian las reglas, cambia el "ganador".

In [ ]:
# Definición de cómputo de métricas y evaluaciones
def simulate_response(question, prompt_version):

    # Simulamos latencia y uso de tokens  en segundos según la versión de prompt
    base_latency = 0.8 if prompt_version == 'prompt_v1' else 1.1 
    noise = random.uniform(-0.15, 0.15)

    # Simulamos latencia con un mínimo de 0.2 segundos
    latency = max(0.2, base_latency + noise)

    # cuenta de tokens de entrada y salida, y costo estimado
    tokens_in = len(question.split()) + (10 if prompt_version == 'prompt_v2' else 6)
    tokens_out = random.randint(40, 80) if prompt_version == 'prompt_v2' else random.randint(20, 60)
    cost = round((tokens_in + tokens_out) * 0.00002, 6)

    
    if prompt_version == 'prompt_v2' and 'regresión' in question.lower():
        accuracy = 1.0 # precisión
        grounded = 1.0 # fundamentación
    else:
        accuracy = random.choice([0.7, 0.8, 0.9]) if prompt_version == 'prompt_v2' else random.choice([0.5, 0.6, 0.7])
        grounded = random.choice([0.7, 0.8, 0.9]) if prompt_version == 'prompt_v2' else random.choice([0.5, 0.6, 0.7])

    # Definimos un error si la precisión es menor a 0.7
    error = None if accuracy >= 0.7 else 'respuesta incompleta'
    response = f'Respuesta simulada ({prompt_version}): {question[:50]}...'

    return {
        'question': question,
        'prompt_version': prompt_version,
        'response': response,
        'latency_s': round(latency, 3),
        'tokens_in': tokens_in,
        'tokens_out': tokens_out,
        'cost_estimated': cost,
        'accuracy': accuracy,
        'groundedness': grounded,
        'error': error,
    }

def evaluate_result(row):
    # Rúbrica simple de calidad basada en accuracy/groundednes para obtener un score de 0 a 100 y decidir que tan buena es la repsuesta
    # criterio 1: precisión ¿ la respuesta es correcta y completa?
    # criterio 2: fundamentación ¿ está basada por fuentes?
    # criterio 3: solo es util si ambos llegan a 0.7
    score = round((row['accuracy'] + row['groundedness']) / 2 * 100)
    return {
        'accuracy': row['accuracy'],
        'groundedness': row['groundedness'],
        'quality_score': score,
        'usable': row['accuracy'] >= 0.7 and row['groundedness'] >= 0.7,
    }

In [ ]:
results = []
for prompt_version in ['prompt_v1', 'prompt_v2']:
    for q in questions:
        row = simulate_response(q, prompt_version)
        metrics = evaluate_result(row)
        results.append({**row, **metrics})

df = pd.DataFrame(results)
display(df.head(20))
# agrupa las métricas por versión de prompt y calcula sumas para el costo total y promedio para la precición y la latencia de tokens
summary = df.groupby('prompt_version').agg({
    'accuracy': 'mean',
    'latency_s': 'mean',
    'tokens_in': 'mean',
    'tokens_out': 'mean',
    'cost_estimated': 'sum',
    'usable': 'sum',
}).rename(columns={'usable': 'usable_count'})
summary['latencia_promedio'] = summary['latency_s'].round(3)
summary['tokens_promedio'] = ((summary['tokens_in'] + summary['tokens_out']) / 2).round(1)
summary['accuracy'] = (summary['accuracy'] * 100).round(1)
summary['costo_total'] = summary['cost_estimated'].round(6)
summary = summary[['accuracy', 'latencia_promedio', 'tokens_promedio', 'costo_total', 'usable_count']]
display(summary)

,question,prompt_version,response,latency_s,tokens_in,tokens_out,cost_estimated,accuracy,groundedness,error,quality_score,usable
0,¿Cuál es la capital de Argentina?,prompt_v1,Respuesta simulada (prompt_v1): ¿Cuál es la ca...,0.923,12,50,0.00124,0.6,0.6,respuesta incompleta,60,False
1,Explícame qué es la observabilidad en LLMs.,prompt_v1,Respuesta simulada (prompt_v1): Explícame qué ...,0.949,13,35,0.00096,0.5,0.7,respuesta incompleta,60,False
2,Calcula 12 * 7.,prompt_v1,Respuesta simulada (prompt_v1): Calcula 12 * 7...,0.899,10,30,0.00080,0.6,0.5,respuesta incompleta,55,False
3,Busca la mejor receta de guacamole.,prompt_v1,Respuesta simulada (prompt_v1): Busca la mejor...,0.935,12,55,0.00134,0.6,0.7,respuesta incompleta,65,False
4,¿Qué hizo Nelson Mandela?,prompt_v1,Respuesta simulada (prompt_v1): ¿Qué hizo Nels...,0.860,10,51,0.00122,0.5,0.5,respuesta incompleta,50,False
5,Resume las ventajas de usar memoria en agentes.,prompt_v1,Respuesta simulada (prompt_v1): Resume las ven...,0.870,14,28,0.00084,0.5,0.7,respuesta incompleta,60,False
6,Define qué es un span en trazabilidad.,prompt_v1,Respuesta simulada (prompt_v1): Define qué es ...,0.825,13,24,0.00074,0.7,0.5,NaN,60,False
7,¿Cómo funciona una batería de auto eléctrico?,prompt_v1,Respuesta simulada (prompt_v1): ¿Cómo funciona...,0.773,13,60,0.00146,0.7,0.7,NaN,70,True
8,Dime un dato sobre el clima actual.,prompt_v1,Respuesta simulada (prompt_v1): Dime un dato s...,0.907,13,31,0.00088,0.7,0.6,NaN,65,False
9,¿Qué es una regresión de prompt?,prompt_v1,Respuesta simulada (prompt_v1): ¿Qué es una re...,0.732,12,22,0.00068,0.7,0.5,NaN,60,False


,accuracy,latencia_promedio,tokens_promedio,costo_total,usable_count
prompt_version,,,,,
prompt_v1,61.0,0.867,25.4,0.01016,1
prompt_v2,85.0,1.084,35.8,0.01432,10


## Interpretación de métricas

- `accuracy` y `groundedness` son métricas de calidad.
- `latencia_s`, `tokens_in` y `tokens_out` son métricas de rendimiento.
- `cost_estimated` ayuda a comparar versiones de prompt desde el punto de vista económico.
- `usable_count` muestra cuántas respuestas cumplen la rúbrica mínima.

Esta comparación ayuda a tomar decisiones cuando un prompt mejorado aumenta la calidad pero también eleva el costo.

## Ejemplo de comparación final

| versión | accuracy (%) | latencia promedio | tokens promedio | costo total | respuestas útiles |
|---|---|---|---|---|---|
| prompt_v1 | valor | valor | valor | valor | valor |
| prompt_v2 | valor | valor | valor | valor | valor |

Esta tabla ejemplifica cómo una métrica compuesta es más útil que la sola apariencia de una respuesta.

: {
: {
: 
3
, 
: 
, 
: 
},
: {
: 
}
: 4,
: 5